# AgentMorph - Stage 3 Dress Rehearsal (smoke test)

200-pair micro-sweep to catch runner/mutator/checker bugs before we burn an Apr 28 overnight on the full 1,000-pair sweep.

**Scope (small on purpose):**
- 2 models: **Llama-3.2-3B** (cheapest) + **Phi-4** (best Stage-1 performer)
- 2 rules: **tool-order-invariance** (no Gemini, simplest) + **refusal-consistency** (Gemini-backed, 3-way compare)
- 1 framework: **native** (smolagents/LangGraph tested in the full sweep)
- 1 env: **ecommerce**

**Expected cell count** after per-rule filtering:
- tool-order-invariance: 2 models x 1 framework x 18 non-refusal scenarios = 36
- refusal-consistency: 2 models x 1 framework x 2 refusal scenarios = 4 cells (each = 3 trajectories)
- **Total: 40 cells, ~50 trajectories**

**Wall-clock on T4:** 25-40 min.

**Pass gate:** `bugs.jsonl` produced, no `[stage3-fail]` lines, `manifest.json` has 40 entries.

**Before you run:** Runtime > Change runtime type > **T4 GPU**. Have your HF token ready for Section 3.
The paraphrase cache (committed to `main` in the previous step) must be on the branch you pull.

## 1. GPU sanity check

In [ ]:
import torch
if torch.cuda.is_available():
    dev = torch.cuda.get_device_properties(0)
    print(f'CUDA OK: {torch.cuda.get_device_name(0)} ({dev.total_memory / 1024**3:.1f} GB VRAM)')
else:
    print('NOTE: no CUDA - pipeline falls back to CPU (very slow). '
          'Dress rehearsal will only complete in a reasonable time on GPU.')

## 2. Clone / pull the repo

In [ ]:
import os
REPO_URL = "https://github.com/Pranamya16/AgentMorph.git"
REPO_DIR = "/content/AgentMorph"
if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git fetch origin && git checkout main && git pull --ff-only
!cd {REPO_DIR} && git log -1 --oneline

## 3. HuggingFace auth

Paste your token, run the cell, then **clear the token** before sharing the notebook.

In [ ]:
from huggingface_hub import login
HF_TOKEN = "hf_REPLACE_ME"
login(token=HF_TOKEN)

## 4. Mount Drive + install extras

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# For the dress rehearsal we only need native + gemini (no smolagents/langgraph).
!pip install -q -e /content/AgentMorph[models,langgraph,gemini]

## 5. Verify the paraphrase cache is populated

Stage 3 calls `paraphrase(..., offline=True)` during the sweep. If the cache isn't on `main`, the Gemini-backed rules raise `ParaphraseCacheMiss` and the whole rule's cells error out.

Expect: 3 files totalling 54 entries (30 + 20 + 4 for rules 2, 3, 5).

In [ ]:
import pathlib
cache_dir = pathlib.Path('/content/AgentMorph/runs/paraphrase_cache')
if not cache_dir.exists():
    raise SystemExit('paraphrase cache not committed to main. '
                     'Re-run notebooks/stage2_seed_paraphrases.ipynb and commit.')
for p in sorted(cache_dir.glob('*.jsonl')):
    n = sum(1 for _ in p.open())
    print(f'{p.name}: {n} entries')

## 6. Dress-rehearsal parameters

Change these to widen/narrow the smoke run. Defaults give ~40 cells / ~30 min.

In [ ]:
OUT_DIR = "/content/drive/MyDrive/AgentMorph/runs/stage3_dress"
HF_CACHE = "/content/drive/MyDrive/AgentMorph/hf_cache"
MODELS = ["Llama-3.2-3B", "Phi-4"]
RULES = ["tool-order-invariance", "refusal-consistency"]
FRAMEWORKS = ["native"]
N_SCENARIOS = 20  # all 20 seed scenarios; rule filters prune as needed
print('dress-rehearsal config:')
print('  models    :', MODELS)
print('  rules     :', RULES)
print('  frameworks:', FRAMEWORKS)

## 7. Dry-run: count the cells that will actually run

No model loaded. Sanity-checks the filter + manifest before we commit GPU time.

In [ ]:
def _flags():
    f = []
    for m in MODELS: f += ['--model', m]
    for r in RULES: f += ['--rule', r]
    for fw in FRAMEWORKS: f += ['--framework', fw]
    return f
flags = ' '.join(_flags())
!cd /content/AgentMorph && python -m agentmorph.runner --stage3 --dry-run \
    {flags} --env ecommerce --n-scenarios {N_SCENARIOS} \
    --out-dir {OUT_DIR}

## 8. Run the dress rehearsal

Resumable: if Colab kills the runtime, re-run this cell and the manifest skips already-completed cells.

In [ ]:
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
flags = ' '.join(_flags())
!cd /content/AgentMorph && PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True \
    python -m agentmorph.runner --stage3 \
    {flags} --env ecommerce --n-scenarios {N_SCENARIOS} \
    --capture-state \
    --hf-cache-dir {HF_CACHE} \
    --out-dir {OUT_DIR}

## 9. Summarize: manifest, bug count, divergence breakdown

In [ ]:
import json, collections, pathlib
out = pathlib.Path(OUT_DIR)
manifest = json.loads((out / 'manifest.json').read_text()) if (out / 'manifest.json').exists() else {'completed': {}}
print(f'manifest cells: {len(manifest.get("completed", {}))}')

bugs_path = out / 'bugs.jsonl'
if bugs_path.exists():
    bugs = [json.loads(l) for l in bugs_path.open()]
    print(f'bugs.jsonl rows: {len(bugs)}')
    by_rule = collections.Counter(b['rule_id'] for b in bugs)
    by_dv = collections.Counter(b['divergence_type'] for b in bugs)
    by_model = collections.Counter(b['model_id'] for b in bugs)
    print('  by rule:         ', dict(by_rule))
    print('  by divergence:   ', dict(by_dv))
    print('  by model:        ', dict(by_model))
else:
    print('no bugs.jsonl yet - either every pair passed or the run was pure error.')

## 10. Peek one pair JSONL line

Useful for visually confirming the pair record contains both `original` + `mutated` trajectories + the `mutation_metadata`.

In [ ]:
import json, pathlib
traj_dir = pathlib.Path(OUT_DIR) / 'trajectories'
for p in sorted(traj_dir.glob('*.jsonl'))[:1]:
    print(f'\n=== {p.name} ===')
    line = p.open().readline()
    if not line:
        print('(empty)')
        continue
    row = json.loads(line)
    print('pair_id          :', row.get('pair_id'))
    print('rule_id          :', row.get('rule_id'))
    print('scenario_id      :', row.get('scenario_id'))
    print('is_bug           :', row.get('is_bug'))
    print('divergence_type  :', row.get('divergence_type'))
    if 'original' in row:
        o = row['original']
        print(f'original: steps={len(o["steps"])} final={str(o["final_answer"])[:80]!r}')
        m = row['mutated']
        print(f'mutated : steps={len(m["steps"])} final={str(m["final_answer"])[:80]!r}')
    elif 'trajectories' in row:
        print(f'3-way trajectories: {len(row["trajectories"])}')
        for i, t in enumerate(row['trajectories']):
            print(f'  [{i}] steps={len(t["steps"])} final={str(t["final_answer"])[:80]!r}')

## 11. Error audit

Count `[stage3-fail]` lines in the runner's stderr. Zero means clean. If non-zero, paste back here so we debug before the full sweep.

In [ ]:
# The dress-rehearsal runner prints stage3-fail lines to stderr.
# They don't land in a file; re-run Section 8 in a fresh cell with
# `2>&1 | tee /tmp/stage3_dress.log` appended if you need a log.
# Also: an error trajectory is one with a final_answer=None and an
# ERROR step. Count those below.
import json, pathlib
traj_dir = pathlib.Path(OUT_DIR) / 'trajectories'
err_pairs = 0
total_pairs = 0
for p in traj_dir.glob('*.jsonl'):
    for line in p.open():
        row = json.loads(line)
        total_pairs += 1
        trajs = row.get('trajectories') or [row.get('original'), row.get('mutated')]
        for t in trajs:
            if t is None: continue
            if t.get('final_answer') is None or any(s.get('kind') == 'error' for s in t.get('steps', [])):
                err_pairs += 1
                break
print(f'pairs with >=1 error trajectory: {err_pairs} / {total_pairs}')

## 12. Gate check

Copy this output to the chat side as part of the dress-rehearsal sign-off. If all gates green, proceed to `stage3_full_sweep.ipynb`.

In [ ]:
gates = {
    'cells_completed': len(manifest.get('completed', {})),
    'bugs_found'     : (bugs_path.exists() and sum(1 for _ in bugs_path.open())) or 0,
    'trajectory_files': sum(1 for _ in (pathlib.Path(OUT_DIR) / 'trajectories').glob('*.jsonl')),
    'error_pair_rate': round(err_pairs / max(total_pairs, 1), 3),
}
print('GATES:')
for k, v in gates.items():
    print(f'  {k:20s} {v}')

ready = (
    gates['cells_completed'] > 0
    and gates['trajectory_files'] >= 1
    and gates['error_pair_rate'] <= 0.20
)
print('\nREADY FOR FULL SWEEP:' , 'YES' if ready else 'NO - debug first')